# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manahilrubabsatti/flyrank-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [2]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Hugging Face connection ready.")

Hugging Face connection ready.


In [3]:
# Check available warehouse months before designing the time split

months = con.sql("""
SELECT
    month,
    COUNT(*) AS rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
GROUP BY month
ORDER BY month
""").df()

display(months)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month,rows
0,2025-01,1297
1,2025-02,75985
2,2025-03,167859
3,2025-04,285114
4,2025-05,349923
5,2025-06,329201
6,2025-07,469794
7,2025-08,704962
8,2025-09,845813
9,2025-10,2165471


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*



I chose Logistic Regression because my Refresh / Content Opportunity Scoring lane needs an interpretable ranking model rather than a complex model for its own sake.

The model estimates the likelihood that a page-client will receive at least one Google Search click in the following month. I use search impressions, clicks, CTR, and average position from the current month as features.

Logistic Regression is suitable for this first model because its coefficients can be inspected and the output probability can be used to rank pages for review.

The goal is not to prove that the model causes better performance. It is to test whether a simple supervised model can provide useful ranking signal beyond my Week-4 baseline.

In [4]:
# ML-08 — Imports and basic setup

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score
)

print("Setup complete.")

Setup complete.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*



I use a time-aware split because the goal is to rank pages using information available at the time of the decision.

Training examples use January 2026 features to predict February outcomes and February 2026 features to predict March outcomes.

The evaluation set uses March 2026 features to predict April 2026 outcomes.

This prevents future information from entering the training features. It also matches the practical workflow: observe a page's current performance, rank it, and then evaluate what happened in the following month.

The Week-4 baseline is also calculated from the March 2026 features and evaluated against the same April 2026 outcome, so the model and baseline are compared on the same candidates and the same metric.

In [5]:
# ML-08 — Build monthly page-client features
# January through April 2026 only

monthly = con.sql("""
SELECT
    month,
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month IN ('2026-01', '2026-02', '2026-03', '2026-04')
  AND gsc_data_available IS TRUE
GROUP BY
    month,
    client_hash_id,
    content_hash_id
HAVING SUM(gsc_impressions) > 0
ORDER BY
    month
""").df()

monthly["ctr"] = (
    monthly["clicks"].astype(float)
    / monthly["impressions"].astype(float)
)

print("Monthly page-client rows:", len(monthly))
print("\nRows by month:")
print(monthly.groupby("month").size())

display(monthly.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Monthly page-client rows: 646601

Rows by month:
month
2026-01    121544
2026-02    153559
2026-03    176738
2026-04    194760
dtype: int64


,month,client_hash_id,content_hash_id,impressions,clicks,avg_position,ctr
0,2026-01,client_8dbf3abdf07569e0,content_3a9f0a8a5ba143a1,3.0,0.0,15.000000,0.000000
1,2026-01,client_8dbf3abdf07569e0,content_7c0b5124241f75f7,1.0,0.0,100.000000,0.000000
2,2026-01,client_08a6a72ff48e62c0,content_8a2caad93ed6f328,379.0,2.0,7.973615,0.005277
3,2026-01,client_08a6a72ff48e62c0,content_824dff8bfeae0de4,19.0,0.0,17.684211,0.000000
4,2026-01,client_e547b89c05043229,content_c4a9f203b6708ddf,2031.0,2.0,5.078779,0.000985


In [6]:
# ML-08 — Create time-aware supervised examples

def make_examples(feature_month, outcome_month):
    features = monthly[
        monthly["month"] == feature_month
    ].copy()

    outcomes = monthly[
        monthly["month"] == outcome_month
    ][
        [
            "client_hash_id",
            "content_hash_id",
            "clicks"
        ]
    ].copy()

    outcomes = outcomes.rename(
        columns={"clicks": "future_clicks"}
    )

    data = features.merge(
        outcomes,
        on=["client_hash_id", "content_hash_id"],
        how="left"
    )

    data["future_clicks"] = data["future_clicks"].fillna(0)

    data["future_click_label"] = (
        data["future_clicks"] > 0
    ).astype(int)

    return data


train_jan = make_examples("2026-01", "2026-02")
train_feb = make_examples("2026-02", "2026-03")

test_mar = make_examples("2026-03", "2026-04")

train = pd.concat(
    [train_jan, train_feb],
    ignore_index=True
)

print("Training rows:", len(train))
print("Evaluation rows:", len(test_mar))

print("\nTraining positive rate:")
print(round(train["future_click_label"].mean(), 4))

print("\nEvaluation positive rate:")
print(round(test_mar["future_click_label"].mean(), 4))

Training rows: 275103
Evaluation rows: 176738

Training positive rate:
0.3759

Evaluation positive rate:
0.3351


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*



I train Logistic Regression on the January→February and February→March examples.

The model is then applied to March 2026 features to produce a probability of receiving at least one click in April 2026.

For comparison, I reconstruct my Week-4 baseline action score using the same March features.

Both approaches rank exactly the same March page-client candidates. I compare their Precision@20 using the same April click outcome.

Precision@20 answers a practical question for my lane: among the 20 highest-priority pages selected by the method, how many actually received at least one click in the following month?

A higher score is useful, but a more complex model is not automatically better if it does not beat the simple baseline.

In [7]:
# ML-08 — Train Logistic Regression

feature_cols = [
    "impressions",
    "clicks",
    "ctr",
    "avg_position"
]

X_train = train[feature_cols].copy()
y_train = train["future_click_label"]

X_test = test_mar[feature_cols].copy()
y_test = test_mar["future_click_label"]

model = Pipeline([
    (
        "scale",
        StandardScaler()
    ),
    (
        "logistic",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

model.fit(X_train, y_train)

test_mar["model_probability"] = model.predict_proba(
    X_test
)[:, 1]

test_mar["model_prediction"] = (
    test_mar["model_probability"] >= 0.5
).astype(int)

print("Model trained.")

Model trained.


In [8]:
# ML-08 — Reconstruct the Week-4 baseline on March 2026

baseline_eval = test_mar.copy()

baseline_eval["ctr_rank"] = (
    baseline_eval["ctr"].rank(pct=True)
)

baseline_eval["position_rank"] = (
    baseline_eval["avg_position"].rank(pct=True)
)

baseline_eval["low_ctr_score"] = (
    1 - baseline_eval["ctr_rank"]
)

baseline_eval["weak_position_score"] = (
    baseline_eval["position_rank"]
)

baseline_eval["visibility_score"] = (
    baseline_eval["impressions"].rank(pct=True)
)

baseline_eval["action_score"] = (
    0.50 * baseline_eval["low_ctr_score"]
    + 0.35 * baseline_eval["weak_position_score"]
    + 0.15 * baseline_eval["visibility_score"]
)

baseline_eval = baseline_eval.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

baseline_eval["baseline_rank"] = (
    np.arange(1, len(baseline_eval) + 1)
)

print("Baseline candidates:", len(baseline_eval))

Baseline candidates: 176738


In [9]:
# ML-08 — Compare model and baseline on the same test set

def precision_at_k(y_true, scores, k=20):
    temp = pd.DataFrame({
        "y_true": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    top_k = temp.sort_values(
        "score",
        ascending=False
    ).head(k)

    return top_k["y_true"].mean()


model_p20 = precision_at_k(
    baseline_eval["future_click_label"],
    baseline_eval["model_probability"],
    k=20
)

baseline_p20 = precision_at_k(
    baseline_eval["future_click_label"],
    baseline_eval["action_score"],
    k=20
)

comparison = pd.DataFrame({
    "method": [
        "W04 baseline",
        "Logistic Regression"
    ],
    "Precision@20": [
        baseline_p20,
        model_p20
    ]
})

display(comparison)

print(
    f"Baseline Precision@20: {baseline_p20:.3f}"
)

print(
    f"Model Precision@20: {model_p20:.3f}"
)

,method,Precision@20
0,W04 baseline,0.15
1,Logistic Regression,0.90


Baseline Precision@20: 0.150
Model Precision@20: 0.900


In [10]:
# ML-08 — Additional model metrics

model_auc = roc_auc_score(
    y_test,
    test_mar["model_probability"]
)

model_ap = average_precision_score(
    y_test,
    test_mar["model_probability"]
)

model_precision = precision_score(
    y_test,
    test_mar["model_prediction"],
    zero_division=0
)

model_recall = recall_score(
    y_test,
    test_mar["model_prediction"],
    zero_division=0
)

metrics = pd.DataFrame({
    "metric": [
        "Precision@20",
        "ROC-AUC",
        "Average Precision",
        "Precision at 0.5 threshold",
        "Recall at 0.5 threshold"
    ],
    "value": [
        model_p20,
        model_auc,
        model_ap,
        model_precision,
        model_recall
    ]
})

display(metrics)

,metric,value
0,Precision@20,0.900000
1,ROC-AUC,0.896469
2,Average Precision,0.855042
3,Precision at 0.5 threshold,0.801093
4,Recall at 0.5 threshold,0.730062


In [11]:
# ML-08 — Logistic Regression feature interpretation

logistic = model.named_steps["logistic"]

coef_table = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": logistic.coef_[0]
})

coef_table["absolute_coefficient"] = (
    coef_table["coefficient"].abs()
)

coef_table = coef_table.sort_values(
    "absolute_coefficient",
    ascending=False
)

display(coef_table)

,feature,coefficient,absolute_coefficient
1,clicks,13.547240,13.547240
0,impressions,5.632222,5.632222
3,avg_position,-0.408020,0.408020
2,ctr,-0.078852,0.078852


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*



The model was evaluated on March 2026 page-client observations using April 2026 clicks as the future outcome.

The model's Precision@20 was **0.900**, compared with **0.150** for the Week-4 baseline.

The model **did improve** on the baseline on this metric. This means that the Logistic Regression model provided stronger top-20 ranking signal on this particular time-aware test.

The coefficient results show that the model relies most on **clicks and impressions**. The standardized coefficients were positive for clicks (13.55) and impressions (5.63), while average position (-0.41) and CTR (-0.08) had much smaller absolute coefficients. These relationships should be interpreted as predictive associations rather than causal effects.

The model produced 10,735 false positives and 15,986 false negatives at the 0.5 probability threshold. The false-positive examples show that some pages had substantial impressions but still received no clicks in April. The false-negative examples show that some pages with weak March performance later received clicks in April.

These errors are expected because future clicks depend on factors not represented in these features, including search intent, SERP context, competition, and changes between months.

Overall, the model showed a much stronger top-20 ranking result than the Week-4 baseline on this test. However, this is one time-aware evaluation period, so the result should be treated as measured evidence rather than proof that the model will always outperform the baseline.

In [12]:
# ML-08 — Error analysis

error_review = baseline_eval.copy()

error_review["model_top20"] = False
error_review.loc[
    error_review["model_probability"].nlargest(20).index,
    "model_top20"
] = True

error_review["baseline_top20"] = False
error_review.loc[
    error_review["action_score"].nlargest(20).index,
    "baseline_top20"
] = True

# False positives/negatives at the 0.5 model threshold
error_review["error_type"] = "CORRECT"

error_review.loc[
    (error_review["model_prediction"] == 1) &
    (error_review["future_click_label"] == 0),
    "error_type"
] = "FALSE_POSITIVE"

error_review.loc[
    (error_review["model_prediction"] == 0) &
    (error_review["future_click_label"] == 1),
    "error_type"
] = "FALSE_NEGATIVE"

print("Model error counts:")
print(error_review["error_type"].value_counts())

print("\nFalse positives:")
display(
    error_review[
        error_review["error_type"] == "FALSE_POSITIVE"
    ][
        [
            "impressions",
            "clicks",
            "ctr",
            "avg_position",
            "model_probability",
            "future_clicks"
        ]
    ].head(10)
)

print("\nFalse negatives:")
display(
    error_review[
        error_review["error_type"] == "FALSE_NEGATIVE"
    ][
        [
            "impressions",
            "clicks",
            "ctr",
            "avg_position",
            "model_probability",
            "future_clicks"
        ]
    ].head(10)
)

Model error counts:
error_type
CORRECT           150017
FALSE_NEGATIVE     15986
FALSE_POSITIVE     10735
Name: count, dtype: int64

False positives:


,impressions,clicks,ctr,avg_position,model_probability,future_clicks
1,21939.0,0.0,0.0,61.379142,1.000000,0.0
2,3309.0,0.0,0.0,74.363856,0.742641,0.0
3,4690.0,0.0,0.0,66.943070,0.956504,0.0
4,11973.0,0.0,0.0,54.937443,0.999998,0.0
6,2693.0,0.0,0.0,76.427033,0.546503,0.0
7,3727.0,0.0,0.0,67.597800,0.857286,0.0
8,10452.0,0.0,0.0,53.172407,0.999985,0.0
11,5822.0,0.0,0.0,57.514256,0.992178,0.0
13,11443.0,0.0,0.0,50.999913,0.999996,0.0
15,2636.0,0.0,0.0,70.834598,0.563986,0.0



False negatives:


,impressions,clicks,ctr,avg_position,model_probability,future_clicks
42,2027.0,0.0,0.0,69.080414,0.376023,1.0
84,1954.0,0.0,0.0,62.578301,0.393389,1.0
98,970.0,0.0,0.0,83.170103,0.092682,1.0
131,1831.0,0.0,0.0,58.884762,0.377582,1.0
140,2029.0,0.0,0.0,56.056678,0.459487,2.0
142,2095.0,0.0,0.0,55.154654,0.487253,1.0
176,776.0,0.0,0.0,80.375000,0.078275,1.0
178,1784.0,0.0,0.0,55.672085,0.382699,1.0
191,1855.0,0.0,0.0,54.028032,0.415664,2.0
250,734.0,0.0,0.0,74.655313,0.085338,1.0


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [13]:
# ML-08 — Final leakage checks

print("FEATURES USED:")
print(feature_cols)

print("\nTIME DESIGN:")
print("Training: 2026-01 -> 2026-02")
print("Training: 2026-02 -> 2026-03")
print("Evaluation: 2026-03 -> 2026-04")

print("\nLEAKAGE CHECKS:")
print("April clicks used as TARGET only: YES")
print("April clicks used as model features: NO")
print("June 2026 sealed test used: NO")
print("Product flags used: NO")
print("Client names used: NO")
print("URLs used: NO")
print("Private queries used: NO")

FEATURES USED:
['impressions', 'clicks', 'ctr', 'avg_position']

TIME DESIGN:
Training: 2026-01 -> 2026-02
Training: 2026-02 -> 2026-03
Evaluation: 2026-03 -> 2026-04

LEAKAGE CHECKS:
April clicks used as TARGET only: YES
April clicks used as model features: NO
June 2026 sealed test used: NO
Product flags used: NO
Client names used: NO
URLs used: NO
Private queries used: NO


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.